In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from scipy.io import savemat

# Designing a suitable distribution

In [ ]:
def _rot_mat (theta, B):
    _theta = np.array(theta)
    return np.moveaxis(np.array([[np.cos(_theta), B*np.sin(_theta)],[-np.sin(_theta)/B, np.cos(_theta)]]),2,0)
def rotation (vec:np.ndarray, theta:float, B:float=1):
    """
    Rotates vector on the plane of an angle theta around an ellipsis with y-oriented axis B times the x-oriented one.
    Input are expected to be np.ndarrays of any shape as long as at leas one dimension ha length two. If more than one
    dimension has length 2, the first is considered to be the spatial dimension.
    """
    if 2 in vec.shape:
        myAx = np.min(np.nonzero(np.array(vec.shape)==2))
        _vec = np.moveaxis(vec, myAx, 0)
        myShape = _vec.shape
        _vec = _vec.reshape((2,-1))
        if np.isscalar(theta):
            rot_mat = _rot_mat(np.full(_vec.shape[1], theta), B)
        else:
            rot_mat = _rot_mat(theta, B)
        res = np.empty_like(_vec)
        for i in range(_vec.shape[1]):
            res[:,i]=_vec[:,i]@rot_mat[i]
        return np.moveaxis(res.reshape(myShape),0,myAx)
    else:
        raise ValueError(f"Invalid input shape {vec.shape}")


In [ ]:
fig, ax = plt.subplots(6,6, sharex="col", sharey="col", figsize=(12,12), squeeze=False)
N = 20000
for mf in range(2,8):
    for a, r in enumerate([0.]+[0.1+0.2*i for i in range(5)]):
        B = np.sqrt((1-r)/(1+r))
        x = np.zeros((2,N))
        x[0,:] = np.sqrt(np.random.chisquare(mf,N))
        x = rotation(rotation(x, 2*np.pi*np.random.random(x.size//2),B),np.pi/4)#0)#
        lab = f"f{mf} c{1-B:.2}\n"+r"$\rho$"+f"{pearsonr(*x)[0]:.4}\nr{r:.2} B{B:.2}"
        plt.sca(ax[mf-2,a])
        plt.hist2d(*x, bins=40, density=True)
        ax[mf-2,a].set_aspect('equal', 'box')
        plt.text(0,0,lab)
plt.subplots_adjust(wspace=0.0, hspace=0.0)
plt.show()

## Creating fake distributions

In [ ]:
P = 1000
for N in [30, 100, 300, 1000, 3000, 10000, 30000]:
    for df in range(2,8):
        for r in [0.]+[0.1+0.2*i for i in range(5)]:
            B = np.sqrt((1-r)/(1+r))
            exp = np.zeros((N,2,P))
            exp[:,0,:] = np.sqrt(np.random.chisquare(df,(N,P)))
            exp = rotation(rotation(exp, 2*np.pi*np.random.random(N*P),B),np.pi/4)
            # print(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}")
            savemat(f"/mnt/DATA/NonLinearMI/verify/crafted_{N}_{df}_r{f'{r:2}'[2]}",{"crafted":exp})
    assert False